# MobileNetV3-Small Hyperparameter Search and Training

이 노트북은 MobileNetV3-Small 기반 OPEN/CLOSE 눈 상태 분류 모델을 학습합니다.

핵심 절차:
- Google Drive의 `eye_data_set.zip` 사용
- Optuna 기반 하이퍼파라미터 탐색
- validation 기준 우선순위별 best 모델 저장
- best F1 설정으로 최종 학습 및 test 평가

주의: test set은 최종 평가에만 사용합니다.

## 1. Environment Setup

필요 라이브러리를 설치하고 GPU 사용 가능 여부를 확인합니다. Optuna는 Bayesian Optimization 계열의 TPE sampler를 사용합니다.

In [ ]:
!pip -q install optuna onnx onnxruntime scikit-learn seaborn

import sys
import torch
import torchvision

print('Python:', sys.version)
print('PyTorch:', torch.__version__)
print('TorchVision:', torchvision.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    !nvidia-smi

## 2. Mount Drive and Locate Dataset

Drive에서 프로젝트 폴더와 `eye_data_set.zip`을 찾습니다. 압축 파일은 `Face_Detection` 루트에 있어도 되고, 하위 폴더에 있어도 자동 탐색합니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import shutil
import subprocess

# 핵심 설정: 데이터셋 zip 파일명과 출력 폴더 이름입니다.
DATA_ZIP_NAME = 'eye_data_set.zip'
PROJECT_FOLDER_CANDIDATES = ['Face_Detection', 'Face Detection', 'FaceDetection']

mydrive = Path('/content/drive/MyDrive')
project_dir = None
for name in PROJECT_FOLDER_CANDIDATES:
    p = mydrive / name
    if p.exists():
        project_dir = p
        break

zip_matches = list(mydrive.glob(f'**/{DATA_ZIP_NAME}'))
if not zip_matches:
    raise FileNotFoundError(f'{DATA_ZIP_NAME} not found under Google Drive MyDrive.')

data_zip = zip_matches[0]
if project_dir is None:
    project_dir = data_zip.parent

src_dir = project_dir / 'src'
src_dir.mkdir(parents=True, exist_ok=True)

output_dir = src_dir / 'outputs' / 'mobilenetv3_optuna'
output_dir.mkdir(parents=True, exist_ok=True)

work_dir = Path('/content/mobilenetv3_eye_state')
extract_dir = work_dir / 'extracted'
work_dir.mkdir(parents=True, exist_ok=True)

print('PROJECT_DIR:', project_dir)
print('SRC_DIR:', src_dir)
print('DATA_ZIP:', data_zip)
print('OUTPUT_DIR:', output_dir)

## 3. Extract Dataset and Validate Structure

압축을 Colab 로컬 디스크에 풀고, `train/val/test` 및 `awake/sleepy` 폴더 구조를 확인합니다.

In [ ]:
SPLITS = ['train', 'val', 'test']
CLASSES = ['awake', 'sleepy']
VALID_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

def is_dataset_root(p: Path) -> bool:
    return all((p / split / cls).exists() for split in SPLITS for cls in CLASSES)

def find_dataset_root(root: Path):
    if is_dataset_root(root):
        return root
    for candidate in root.rglob('*'):
        if candidate.is_dir() and is_dataset_root(candidate):
            return candidate
    return None

if not find_dataset_root(extract_dir):
    if extract_dir.exists():
        shutil.rmtree(extract_dir)
    extract_dir.mkdir(parents=True, exist_ok=True)
    print('Extracting dataset...')
    subprocess.run(['unzip', '-q', '-o', str(data_zip), '-d', str(extract_dir)], check=True)

data_dir = find_dataset_root(extract_dir)
if data_dir is None:
    raise FileNotFoundError('Could not find train/val/test awake/sleepy folders after extraction.')

counts = {}
for split in SPLITS:
    counts[split] = {}
    for cls in CLASSES:
        folder = data_dir / split / cls
        counts[split][cls] = sum(1 for p in folder.iterdir() if p.is_file() and p.suffix.lower() in VALID_EXTS)

print('DATA_DIR:', data_dir)
counts

## 4. Search Configuration

탐색 예산은 10 trials, trial당 5 epochs입니다. 앞 5 trials는 random startup 탐색, 이후 5 trials는 TPE 기반 Bayesian 탐색으로 진행합니다.

In [ ]:
import json
import random
import time
from collections import defaultdict

import numpy as np
import pandas as pd
import optuna
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    average_precision_score,
    roc_auc_score,
    confusion_matrix,
)

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights

SEED = 42
N_TRIALS = 10
N_STARTUP_TRIALS = 5
TRIAL_EPOCHS = 5
FINAL_EPOCHS = 30
NUM_WORKERS = 2
POSITIVE_CLASS_NAME = 'sleepy'  # sleepy 폴더를 CLOSE 클래스로 해석합니다.

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
PIN_MEMORY = DEVICE.type == 'cuda'

print('DEVICE:', DEVICE)

## 5. DataLoader and Metric Functions

각 trial마다 `img_size`, `batch_size`가 달라질 수 있으므로 DataLoader를 매번 새로 생성합니다. 평가 기준은 CLOSE class 기준 precision/recall/F1/PR-AUC입니다.

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

def build_loaders(img_size: int, batch_size: int):
    train_tfms = transforms.Compose([
        transforms.Grayscale(num_output_channels=3),
        transforms.Resize((img_size, img_size)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(degrees=8),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])
    eval_tfms = transforms.Compose([
        transforms.Grayscale(num_output_channels=3),
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])

    train_ds = datasets.ImageFolder(data_dir / 'train', transform=train_tfms)
    val_ds = datasets.ImageFolder(data_dir / 'val', transform=eval_tfms)
    test_ds = datasets.ImageFolder(data_dir / 'test', transform=eval_tfms)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
    return train_ds, val_ds, test_ds, train_loader, val_loader, test_loader


def build_model(dropout: float, num_classes: int):
    weights = MobileNet_V3_Small_Weights.DEFAULT
    model = mobilenet_v3_small(weights=weights)
    if isinstance(model.classifier[2], nn.Dropout):
        model.classifier[2] = nn.Dropout(p=dropout, inplace=True)
    in_features = model.classifier[-1].in_features
    model.classifier[-1] = nn.Linear(in_features, num_classes)
    return model


def run_epoch(model, loader, criterion, optimizer=None, scaler=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    total_loss = 0.0
    total = 0
    use_amp = DEVICE.type == 'cuda'

    for images, labels in loader:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        with torch.set_grad_enabled(is_train):
            with torch.cuda.amp.autocast(enabled=use_amp):
                logits = model(images)
                loss = criterion(logits, labels)
            if is_train:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
        batch = labels.size(0)
        total_loss += loss.item() * batch
        total += batch
    return total_loss / max(total, 1)


def predict_probs(model, loader):
    model.eval()
    y_true, y_prob = [], []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(DEVICE, non_blocking=True)
            logits = model(images)
            probs = torch.softmax(logits, dim=1).cpu().numpy()
            y_prob.append(probs)
            y_true.extend(labels.numpy().tolist())
    return np.array(y_true), np.concatenate(y_prob, axis=0)


def calc_metrics(y_true, probs, positive_idx):
    y_pred = probs.argmax(axis=1)
    pos_prob = probs[:, positive_idx]
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=[positive_idx], average='binary', pos_label=positive_idx, zero_division=0
    )
    out = {
        'accuracy': float(accuracy_score(y_true, y_pred)),
        'close_precision': float(precision),
        'close_recall': float(recall),
        'f1': float(f1),
        'pr_auc': float(average_precision_score((np.array(y_true) == positive_idx).astype(int), pos_prob)),
    }
    try:
        out['roc_auc'] = float(roc_auc_score((np.array(y_true) == positive_idx).astype(int), pos_prob))
    except ValueError:
        out['roc_auc'] = float('nan')
    return out


def save_json(path, data):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, indent=2), encoding='utf-8')

## 6. Optuna Objective

각 trial은 하이퍼파라미터를 샘플링하고 짧게 학습한 뒤 validation F1-score를 반환합니다. 동시에 우선순위별 최고 모델을 Drive에 저장합니다.

In [ ]:
priority_specs = {
    'best_by_close_recall': 'close_recall',
    'best_by_f1': 'f1',
    'best_by_pr_auc': 'pr_auc',
    'best_by_accuracy': 'accuracy',
}
best_scores = {name: -1.0 for name in priority_specs}
trial_rows = []

def save_priority_model(priority_name, model, config, metrics, class_to_idx):
    target = output_dir / priority_name
    target.mkdir(parents=True, exist_ok=True)
    torch.save({
        'arch': 'mobilenet_v3_small',
        'model_state': model.state_dict(),
        'config': config,
        'metrics': metrics,
        'class_to_idx': class_to_idx,
        'positive_class_name': POSITIVE_CLASS_NAME,
        'imagenet_mean': IMAGENET_MEAN,
        'imagenet_std': IMAGENET_STD,
    }, target / 'model.pt')
    save_json(target / 'config.json', config)
    save_json(target / 'metrics.json', metrics)


def objective(trial):
    img_size = trial.suggest_categorical('img_size', [96, 128, 160])
    batch_size = trial.suggest_categorical('batch_size', [64, 128])
    lr = trial.suggest_float('learning_rate', 1e-4, 1e-2, log=True)
    weight_decay = trial.suggest_float('weight_decay', 1e-5, 1e-3, log=True)
    dropout = trial.suggest_float('dropout', 0.0, 0.3)

    config = {
        'img_size': img_size,
        'batch_size': batch_size,
        'learning_rate': lr,
        'weight_decay': weight_decay,
        'dropout': dropout,
        'trial_epochs': TRIAL_EPOCHS,
    }

    train_ds, val_ds, _, train_loader, val_loader, _ = build_loaders(img_size, batch_size)
    positive_idx = train_ds.class_to_idx[POSITIVE_CLASS_NAME]
    model = build_model(dropout, num_classes=len(train_ds.classes)).to(DEVICE)

    targets = torch.tensor([label for _, label in train_ds.samples], dtype=torch.long)
    counts = torch.bincount(targets, minlength=len(train_ds.classes)).float()
    class_weights = counts.sum() / (len(train_ds.classes) * counts.clamp(min=1))
    criterion = nn.CrossEntropyLoss(weight=class_weights.to(DEVICE))
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scaler = torch.cuda.amp.GradScaler(enabled=DEVICE.type == 'cuda')

    for epoch in range(TRIAL_EPOCHS):
        train_loss = run_epoch(model, train_loader, criterion, optimizer, scaler)
        val_loss = run_epoch(model, val_loader, criterion)
        y_true, probs = predict_probs(model, val_loader)
        metrics = calc_metrics(y_true, probs, positive_idx)
        trial.report(metrics['f1'], epoch)
        if trial.should_prune():
            raise optuna.TrialPruned()

    metrics.update({'val_loss': float(val_loss), 'train_loss': float(train_loss), 'trial': trial.number})
    row = {**config, **metrics, 'trial': trial.number}
    trial_rows.append(row)
    pd.DataFrame(trial_rows).to_csv(output_dir / 'tuning_results.csv', index=False)

    for priority_name, metric_name in priority_specs.items():
        score = metrics[metric_name]
        if score > best_scores[priority_name]:
            best_scores[priority_name] = score
            save_priority_model(priority_name, model, {**config, 'trial': trial.number}, metrics, train_ds.class_to_idx)

    return metrics['f1']

## 7. Run Search

여기서 실제 탐색이 진행됩니다. Colab이 중간에 끊겨도 SQLite study DB와 CSV가 Drive에 저장됩니다.

In [ ]:
study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=SEED, n_startup_trials=N_STARTUP_TRIALS),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=3, n_warmup_steps=2),
    study_name='mobilenetv3_eye_state',
    storage=f"sqlite:///{output_dir / 'optuna_study.db'}",
    load_if_exists=True,
)
study.optimize(objective, n_trials=N_TRIALS)

print('Best trial:', study.best_trial.number)
print('Best value:', study.best_value)
print('Best params:', study.best_params)

## 8. Final Training with Best F1 Config

탐색에서 validation F1-score가 가장 좋았던 설정으로 더 길게 최종 학습합니다.

In [ ]:
best_params = study.best_params
final_config = {
    'img_size': best_params['img_size'],
    'batch_size': best_params['batch_size'],
    'learning_rate': best_params['learning_rate'],
    'weight_decay': best_params['weight_decay'],
    'dropout': best_params['dropout'],
    'final_epochs': FINAL_EPOCHS,
}

train_ds, val_ds, test_ds, train_loader, val_loader, test_loader = build_loaders(final_config['img_size'], final_config['batch_size'])
positive_idx = train_ds.class_to_idx[POSITIVE_CLASS_NAME]
model = build_model(final_config['dropout'], num_classes=len(train_ds.classes)).to(DEVICE)

targets = torch.tensor([label for _, label in train_ds.samples], dtype=torch.long)
counts = torch.bincount(targets, minlength=len(train_ds.classes)).float()
class_weights = counts.sum() / (len(train_ds.classes) * counts.clamp(min=1))
criterion = nn.CrossEntropyLoss(weight=class_weights.to(DEVICE))
optimizer = torch.optim.AdamW(model.parameters(), lr=final_config['learning_rate'], weight_decay=final_config['weight_decay'])
scaler = torch.cuda.amp.GradScaler(enabled=DEVICE.type == 'cuda')

best_val_f1 = -1.0
history = []
final_dir = output_dir / 'final_best_by_f1'
final_dir.mkdir(parents=True, exist_ok=True)

for epoch in range(1, FINAL_EPOCHS + 1):
    train_loss = run_epoch(model, train_loader, criterion, optimizer, scaler)
    val_loss = run_epoch(model, val_loader, criterion)
    y_true, probs = predict_probs(model, val_loader)
    val_metrics = calc_metrics(y_true, probs, positive_idx)
    row = {'epoch': epoch, 'train_loss': train_loss, 'val_loss': val_loss, **val_metrics}
    history.append(row)
    print(row)

    if val_metrics['f1'] > best_val_f1:
        best_val_f1 = val_metrics['f1']
        torch.save({
            'arch': 'mobilenet_v3_small',
            'model_state': model.state_dict(),
            'config': final_config,
            'metrics': val_metrics,
            'class_to_idx': train_ds.class_to_idx,
            'positive_class_name': POSITIVE_CLASS_NAME,
            'imagenet_mean': IMAGENET_MEAN,
            'imagenet_std': IMAGENET_STD,
        }, final_dir / 'model.pt')
        save_json(final_dir / 'config.json', final_config)
        save_json(final_dir / 'val_metrics.json', val_metrics)

pd.DataFrame(history).to_csv(final_dir / 'training_history.csv', index=False)
print('Final best val F1:', best_val_f1)

## 9. Test Evaluation and ONNX Export

최종 모델은 test set에서 한 번만 평가합니다. 이후 Jetson 변환을 위해 ONNX도 저장합니다.

In [ ]:
ckpt = torch.load(final_dir / 'model.pt', map_location=DEVICE)
model = build_model(ckpt['config']['dropout'], num_classes=len(train_ds.classes)).to(DEVICE)
model.load_state_dict(ckpt['model_state'])
model.eval()

y_true, probs = predict_probs(model, test_loader)
test_metrics = calc_metrics(y_true, probs, positive_idx)
y_pred = probs.argmax(axis=1)
cm = confusion_matrix(y_true, y_pred)

save_json(final_dir / 'test_metrics.json', test_metrics)
print('Test metrics:', test_metrics)

plt.figure(figsize=(4, 3))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=train_ds.classes, yticklabels=train_ds.classes, cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('MobileNetV3 Confusion Matrix')
plt.tight_layout()
plt.savefig(final_dir / 'confusion_matrix.png', dpi=150)
plt.show()

model.cpu().eval()
dummy = torch.randn(1, 3, final_config['img_size'], final_config['img_size'])
torch.onnx.export(
    model,
    dummy,
    str(final_dir / 'model.onnx'),
    input_names=['input'],
    output_names=['logits'],
    opset_version=17,
    dynamic_axes={'input': {0: 'batch'}, 'logits': {0: 'batch'}},
)
print('Saved:', final_dir)